# B.7 — Spatial null testing on benchmark data

Applies the BrainSMASH-style spatial null correction from `SpatialPeeler/spatial_nulls.py`
to a single easy benchmark condition:

| Parameter | Value |
|-----------|-------|
| `PERTURB_FRAC` | 0.70 |
| `FIXED_LAM` | 0.5 |
| `TOP_GENES` | 10 |

**Question:** does correcting for spatial autocorrelation reduce the number of
"significant" factors reported by SpatialPeeler, and if so, are the factors
that lose significance the ones *without* a true disease signal?

In [ ]:
import sys
import warnings
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import scipy.sparse as sp
import scanpy as sc
import statsmodels.api as sm
from sklearn.decomposition import NMF
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import roc_auc_score

warnings.simplefilter('ignore', category=ConvergenceWarning)
sc.settings.verbosity = 0

ROOT = Path('/lustre/scratch126/gengen/teams_v2/marks/dp31/SpatialPeeler')
sys.path.insert(0, str(ROOT))

from SpatialPeeler.spatial_nulls import spatial_factor_pvalues_all, coords_from_adata

RAND_SEED  = 28
N_FACTORS  = 15
np.random.seed(RAND_SEED)

DATA_DIR  = ROOT / 'benchmark' / 'generated_benchmark_data'
FINAL_DIR = ROOT / 'benchmark' / 'generated_benchmark_data_final'

# ---- condition ----
PERTURB_FRAC = 0.70
FIXED_LAM    = 0.5
TOP_GENES    = 10
N_SURROGATES = 200   # increase to 500+ for publication-quality p-values

TAG      = f'frac{PERTURB_FRAC:.0%}_lam{FIXED_LAM}_top{TOP_GENES}genes'
CASE_PATH = FINAL_DIR / f'adata06_bot_case_nmfGenes_{TAG}.h5ad'
print('Condition tag:', TAG)
print('Case file:', CASE_PATH)

#### 1. Load data

In [ ]:
adata_ctrl = ad.read_h5ad(DATA_DIR / 'adata06_top.h5ad')
adata_ctrl.obs['sample_id'] = 'ctrl'
adata_ctrl.obs['status']    = 0
adata_ctrl.obs['Condition'] = 'Control'
if 'in_circle' not in adata_ctrl.obs.columns:
    adata_ctrl.obs['in_circle'] = False
adata_ctrl.obs['in_circle'] = adata_ctrl.obs['in_circle'].astype(bool)

adata_case = ad.read_h5ad(CASE_PATH)
adata_case.obs['sample_id'] = 'case'
adata_case.obs['status']    = 1
adata_case.obs['Condition'] = 'Case'
adata_case.obs['in_circle'] = adata_case.obs['in_circle'].astype(bool)

gt_genes = set(
    pd.read_csv(FINAL_DIR / f'nmf_genes_top{TOP_GENES}_per_factor.csv')['gene'].unique()
)

print(f'Control: {adata_ctrl.shape}  |  Case: {adata_case.shape}')
print(f'In-circle beads (case): {adata_case.obs["in_circle"].sum():,}')
print(f'Ground-truth perturbed genes: {len(gt_genes)}')

#### 2. Preprocess + NMF (shared helpers from B.6)

In [ ]:
def preprocess(adata_ctrl, adata_case, force_genes=None):
    adata = ad.concat(
        [adata_ctrl, adata_case],
        join='inner', merge='first',
        label='sample_id', keys=['ctrl', 'case'],
        index_unique='-'
    )
    for col in ['status', 'Condition', 'in_circle']:
        adata.obs[col] = np.concatenate([
            adata_ctrl.obs[col].values,
            adata_case.obs[col].values
        ])
    adata.obs['status']    = adata.obs['status'].astype(int)
    adata.obs['in_circle'] = adata.obs['in_circle'].astype(bool)

    adata.layers['counts'] = adata.X.copy()
    min_cells = max(1, adata.n_obs // 500)
    n_expr    = np.array((adata.X > 0).sum(axis=0)).flatten()
    adata     = adata[:, n_expr >= min_cells].copy()

    adata.layers['lognorm'] = adata.layers['counts'].copy()
    sc.pp.normalize_total(adata, target_sum=1e4, layer='lognorm')
    sc.pp.log1p(adata, layer='lognorm')
    sc.pp.highly_variable_genes(
        adata, n_top_genes=2000, batch_key='sample_id',
        flavor='seurat', layer='lognorm', subset=False
    )
    if force_genes is not None:
        symbols = (adata.var['features'].values
                   if 'features' in adata.var.columns else adata.var_names.values)
        adata.var['highly_variable'] |= np.isin(symbols, list(force_genes))

    adata = adata[:, adata.var['highly_variable']].copy()
    sc.pp.scale(adata, zero_center=False)
    return adata


def run_nmf(adata, n_factors=N_FACTORS, rand_seed=RAND_SEED):
    X = adata.X
    if sp.issparse(X):
        X = X.toarray()
    X = X.astype(np.float32)
    model = NMF(n_components=n_factors, init='nndsvda',
                random_state=rand_seed, max_iter=1000, solver='cd')
    W = model.fit_transform(X)
    adata.obsm['X_nmf'] = W
    return W


def run_spatialpeeler(adata, n_factors=N_FACTORS):
    X  = adata.obsm['X_nmf']
    y  = adata.obs['status'].values
    results = []
    for i in range(n_factors):
        Xi = X[:, i].reshape(-1, 1)
        X_int = sm.add_constant(Xi)
        try:
            fit   = sm.Logit(y, X_int).fit(disp=False)
            coef  = float(fit.params[1])
            pval  = float(fit.pvalues[1])
            p_hat = fit.predict(X_int)
        except Exception:
            coef, pval, p_hat = 0.0, 1.0, np.full(len(y), np.nan)
        results.append({'factor_index': i, 'coef': coef, 'pval': pval, 'p_hat': p_hat})
    return results


print('Preprocessing...')
adata = preprocess(adata_ctrl, adata_case, force_genes=gt_genes)
print('Running NMF...')
run_nmf(adata)
print(f'Combined adata: {adata.shape}')

#### 3. Standard SpatialPeeler (logistic regression)

In [ ]:
print('Running standard SpatialPeeler...')
sp_results = run_spatialpeeler(adata)

sp_df = pd.DataFrame([{
    'factor':      f"F{r['factor_index']+1}",
    'factor_index': r['factor_index'],
    'coef':        r['coef'],
    'standard_pval': r['pval'],
} for r in sp_results])

# AUROC on case beads (in-circle = ground truth)
case_mask = adata.obs['status'].values == 1
gt_vec    = adata.obs['in_circle'].values[case_mask].astype(int)
for r in sp_results:
    ph = r['p_hat'][case_mask]
    try:
        sp_df.loc[sp_df['factor_index'] == r['factor_index'], 'auroc'] = roc_auc_score(gt_vec, ph)
    except Exception:
        sp_df.loc[sp_df['factor_index'] == r['factor_index'], 'auroc'] = np.nan

top_factor = sp_df.sort_values('coef', ascending=False).iloc[0]
print(f"Top factor: {top_factor['factor']}  coef={top_factor['coef']:.3f}  "
      f"pval={top_factor['standard_pval']:.2e}  AUROC={top_factor['auroc']:.3f}")

print(f"\nStandard p<0.05: {(sp_df['standard_pval'] < 0.05).sum()} / {len(sp_df)} factors")
sp_df.sort_values('coef', ascending=False)

#### 4a. Subsample beads for spatial null testing

The BrainSMASH surrogate generator runs a KNN query per delta per surrogate.
At n ≈ 80 k spots the largest neighbourhood (δ = 0.40 → k ≈ 32 k) takes ~115 s
per query → **~13 days** for 200 surrogates × 15 factors.

A stratified subsample of **5,000 beads** preserves the spatial autocorrelation
structure for variogram estimation and brings runtime to **~3 hours**.
NMF factor loadings come from the full-data decomposition; only the beads fed to
the surrogate generator are reduced.

In [ ]:
N_SUBSAMPLE = 5000  # adjust up for more statistical power; down for faster runs

# Build full coordinate array first (needed for both subsampling and Panel A/D)
coords = coords_from_adata(adata, obsm_key='spatial',
                            x_col='Raw_Slideseq_X', y_col='Raw_Slideseq_Y')
adata.obsm['spatial'] = coords
print(f'Full coordinates: {coords.shape}  '
      f'x-range [{coords[:,0].min():.0f}, {coords[:,0].max():.0f}]')

# Stratified subsample: preserve ctrl/case ratio
rng_sub  = np.random.default_rng(RAND_SEED)
ctrl_idx = np.where(adata.obs['status'].values == 0)[0]
case_idx = np.where(adata.obs['status'].values == 1)[0]

n_ctrl_sub = round(N_SUBSAMPLE * len(ctrl_idx) / len(adata))
n_case_sub = N_SUBSAMPLE - n_ctrl_sub
sub_ctrl   = rng_sub.choice(ctrl_idx, n_ctrl_sub, replace=False)
sub_case   = rng_sub.choice(case_idx, n_case_sub, replace=False)
sub_idx    = np.sort(np.concatenate([sub_ctrl, sub_case]))

adata_sub = adata[sub_idx].copy()

print(f'Subsampled: {adata_sub.n_obs:,} spots  '
      f'(ctrl={n_ctrl_sub:,}, case={n_case_sub:,})')
print(f'In-circle fraction — full: {adata.obs["in_circle"].mean():.3f}  '
      f'subsample: {adata_sub.obs["in_circle"].mean():.3f}')

#### 4. Spatial null testing (BrainSMASH-style surrogates)

For each factor, `N_SURROGATES` surrogate factor maps are generated that preserve the
spatial autocorrelation structure but randomise the case/control association.  The
fraction of surrogates with |coef| ≥ |observed coef| is the spatial p-value.

In [ ]:
print(f'Running spatial null test on {adata_sub.n_obs:,} beads '
      f'({N_SURROGATES} surrogates × {N_FACTORS} factors)...')
print('Estimated runtime: ~3 hours\n')

spatial_df = spatial_factor_pvalues_all(
    adata_sub,
    factor_key='X_nmf',
    status_key='status',
    coords_obsm='spatial',
    n_surrogates=N_SURROGATES,
    seed=RAND_SEED,
    verbose=True,
)
spatial_df['factor'] = [f'F{i+1}' for i in spatial_df['factor_index']]
print('\nDone.')
spatial_df

#### 5. Merge results

In [ ]:
compare_df = sp_df.merge(spatial_df[['factor_index', 'spatial_pval']], on='factor_index')
compare_df['neg_log10_standard'] = -np.log10(compare_df['standard_pval'].clip(1e-300))
compare_df['neg_log10_spatial']  = -np.log10(compare_df['spatial_pval'].clip(1/(N_SURROGATES+1)))
compare_df['sig_standard'] = compare_df['standard_pval'] < 0.05
compare_df['sig_spatial']  = compare_df['spatial_pval']  < 0.05
compare_df['top_factor']   = compare_df['factor_index'] == int(top_factor['factor_index'])

print('=== Significance summary ===')
print(f"Standard  p < 0.05:  {compare_df['sig_standard'].sum()} factors")
print(f"Spatial   p < 0.05:  {compare_df['sig_spatial'].sum()} factors")
print()
print(compare_df[['factor', 'coef', 'standard_pval', 'spatial_pval', 'auroc',
                  'sig_standard', 'sig_spatial']].sort_values('coef', ascending=False).to_string(index=False))

#### 6. Visualise results

In [ ]:
# ── Panel A: spatial map of top factor loadings ──────────────────────────────
top_fi     = int(top_factor['factor_index'])
loadings   = adata.obsm['X_nmf'][:, top_fi]
is_case    = adata.obs['status'].values == 1

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, mask, title in [
    (axes[0], ~is_case, 'Control'),
    (axes[1],  is_case, 'Case'),
]:
    sc_ = ax.scatter(
        coords[mask, 0], coords[mask, 1],
        c=loadings[mask], cmap='viridis', s=1, alpha=0.6, rasterized=True
    )
    ax.set_aspect('equal')
    ax.set_title(f'{title} — {top_factor["factor"]} loading', fontsize=11)
    ax.set_xlabel('X', fontsize=9)
    ax.set_ylabel('Y', fontsize=9)
    plt.colorbar(sc_, ax=ax, label='NMF loading', fraction=0.046, pad=0.04)

# overlay in-circle beads on case panel
in_circ = adata.obs['in_circle'].values
axes[1].scatter(
    coords[is_case & in_circ, 0], coords[is_case & in_circ, 1],
    s=3, facecolors='none', edgecolors='red', linewidths=0.4,
    alpha=0.4, label='in-circle (GT)', rasterized=True
)
axes[1].legend(fontsize=8, frameon=False, markerscale=3)

fig.suptitle(f'Spatial distribution of {top_factor["factor"]} ({TAG})', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Panel B: coefficients with significance colouring (standard vs spatial) ──
order    = compare_df.sort_values('factor_index')['factor_index'].values
x_ticks  = np.arange(N_FACTORS)
labels   = [f'F{i+1}' for i in order]

def bar_colors(df, pval_col, alpha_thresh=0.05):
    colors = []
    for _, row in df.sort_values('factor_index').iterrows():
        if row[pval_col] < alpha_thresh and row['coef'] > 0:
            colors.append('#d62728')   # significant positive → red
        elif row[pval_col] < alpha_thresh and row['coef'] < 0:
            colors.append('#1f77b4')   # significant negative → blue
        else:
            colors.append('#aec7e8')   # n.s. → light blue
    return colors

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)

for ax, pval_col, title in [
    (axes[0], 'standard_pval', 'Standard logistic regression p-value'),
    (axes[1], 'spatial_pval',  f'Spatial null p-value ({N_SURROGATES} surrogates)'),
]:
    sub    = compare_df.sort_values('factor_index')
    coefs  = sub['coef'].values
    colors = bar_colors(sub, pval_col)

    ax.bar(x_ticks, coefs, color=colors, edgecolor='black', linewidth=0.6)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels(labels, fontsize=9, rotation=45, ha='right')
    ax.set_xlabel('NMF factor', fontsize=10)
    ax.set_title(title, fontsize=10)

    # mark top factor
    ax.bar(top_fi, coefs[top_fi], color=colors[top_fi],
           edgecolor='gold', linewidth=2.0)

    from matplotlib.patches import Patch
    handles = [
        Patch(facecolor='#d62728', edgecolor='black', label='sig. positive (p<0.05)'),
        Patch(facecolor='#1f77b4', edgecolor='black', label='sig. negative (p<0.05)'),
        Patch(facecolor='#aec7e8', edgecolor='black', label='n.s.'),
        Patch(facecolor='gray',    edgecolor='gold',   linewidth=2, label='top factor'),
    ]
    ax.legend(handles=handles, fontsize=8, frameon=False)

axes[0].set_ylabel('Logistic regression coefficient', fontsize=10)
fig.suptitle(f'SpatialPeeler factor significance — {TAG}', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Panel C: standard −log10(p) vs spatial −log10(p) scatter ─────────────────
fig, ax = plt.subplots(figsize=(5, 5))

x = compare_df['neg_log10_standard'].values
y = compare_df['neg_log10_spatial'].values

# color by AUROC to see whether high-AUROC factors survive the correction
sc_ = ax.scatter(x, y, c=compare_df['auroc'].values,
                 cmap='RdYlGn', vmin=0.5, vmax=1.0, s=60, edgecolors='black',
                 linewidths=0.6, zorder=3)
plt.colorbar(sc_, ax=ax, label='AUROC (in-circle)', fraction=0.046, pad=0.04)

# identity line
lim = max(x.max(), y.max()) * 1.05
ax.plot([0, lim], [0, lim], 'k--', linewidth=0.8, label='y = x')

# significance thresholds
thresh_std  = -np.log10(0.05)
thresh_spat = -np.log10(0.05)
ax.axvline(thresh_std,  color='steelblue', linestyle=':', linewidth=1, label='standard p=0.05')
ax.axhline(thresh_spat, color='tomato',    linestyle=':', linewidth=1, label='spatial p=0.05')

# label each point
for _, row in compare_df.iterrows():
    ax.annotate(row['factor'],
                (row['neg_log10_standard'], row['neg_log10_spatial']),
                fontsize=7, ha='left', va='bottom',
                xytext=(3, 2), textcoords='offset points')

ax.set_xlabel('Standard  −log₁₀(p)', fontsize=10)
ax.set_ylabel(f'Spatial null  −log₁₀(p)  ({N_SURROGATES} surrogates)', fontsize=10)
ax.set_title('Standard vs spatial significance per factor', fontsize=11)
ax.legend(fontsize=8, frameon=False)
ax.set_xlim(left=0)
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.show()

# quadrant summary
both_sig    = (compare_df['sig_standard'] & compare_df['sig_spatial']).sum()
std_only    = (compare_df['sig_standard'] & ~compare_df['sig_spatial']).sum()
spatial_only= (~compare_df['sig_standard'] & compare_df['sig_spatial']).sum()
neither     = (~compare_df['sig_standard'] & ~compare_df['sig_spatial']).sum()

print('Quadrant summary (p < 0.05):')
print(f'  Sig in both:          {both_sig}')
print(f'  Standard only (false positives?): {std_only}')
print(f'  Spatial only:         {spatial_only}')
print(f'  Neither:              {neither}')

In [ ]:
# ── Panel D: null distribution for the top factor ────────────────────────────
# Uses the same subsampled beads as the spatial null test for consistency.
from SpatialPeeler.spatial_nulls import spatial_factor_pvalue

coords_sub   = adata_sub.obsm['spatial']
loadings_sub = adata_sub.obsm['X_nmf'][:, top_fi]
status_sub   = adata_sub.obs['status'].values

print(f'Generating null distribution for {top_factor["factor"]} '
      f'({adata_sub.n_obs:,} subsampled beads)...')
p_val, obs_coef, null_coefs = spatial_factor_pvalue(
    loadings_sub, coords_sub, status_sub,
    n_surrogates=N_SURROGATES, seed=RAND_SEED
)
print(f'Observed coef: {obs_coef:.3f}   Spatial p-value: {p_val:.4f}')

fig, ax = plt.subplots(figsize=(6, 4))
ax.hist(null_coefs, bins=30, color='steelblue', alpha=0.7,
        edgecolor='white', linewidth=0.5, label=f'Null ({N_SURROGATES} surrogates)')
ax.axvline(obs_coef,  color='red',    linewidth=2,   label=f'Observed coef = {obs_coef:.2f}')
ax.axvline(-obs_coef, color='red',    linewidth=1.2, linestyle='--', label='−|observed|')
ax.axvline(np.percentile(np.abs(null_coefs), 95),
           color='orange', linewidth=1.2, linestyle=':', label='95th pct |null|')

ax.set_xlabel('Logistic regression coefficient', fontsize=10)
ax.set_ylabel('Count', fontsize=10)
ax.set_title(f'Null distribution — {top_factor["factor"]}  (spatial p = {p_val:.3f})', fontsize=11)
ax.legend(fontsize=8, frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
# ── Panel E: AUROC grouped by spatial significance ───────────────────────────
fig, ax = plt.subplots(figsize=(5, 4))

groups  = ['sig (both)', 'standard only', 'spatial only', 'n.s.']
def assign_group(row):
    s, sp = row['sig_standard'], row['sig_spatial']
    if s and sp:   return 'sig (both)'
    if s and not sp: return 'standard only'
    if not s and sp: return 'spatial only'
    return 'n.s.'

compare_df['sig_group'] = compare_df.apply(assign_group, axis=1)

rng = np.random.default_rng(RAND_SEED)
palette = {'sig (both)': '#2ca02c', 'standard only': '#ff7f0e',
           'spatial only': '#9467bd', 'n.s.': '#aec7e8'}

present_groups = [g for g in groups if g in compare_df['sig_group'].values]
for xi, grp in enumerate(present_groups):
    vals   = compare_df.loc[compare_df['sig_group'] == grp, 'auroc'].dropna().values
    jitter = rng.uniform(-0.15, 0.15, size=len(vals))
    ax.scatter(np.full_like(vals, xi) + jitter, vals,
               color=palette[grp], s=50, edgecolors='black', linewidths=0.5, zorder=3)
    if len(vals):
        ax.hlines(vals.mean(), xi - 0.3, xi + 0.3, colors='black', linewidth=1.5, zorder=4)

ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.8, label='random')
ax.set_xticks(range(len(present_groups)))
ax.set_xticklabels(present_groups, fontsize=9)
ax.set_ylabel('AUROC (in-circle, case beads)', fontsize=10)
ax.set_title('Factor AUROC by significance category', fontsize=11)
ax.legend(fontsize=8, frameon=False)
plt.tight_layout()
plt.show()

print(compare_df[['factor', 'coef', 'standard_pval', 'spatial_pval', 'auroc', 'sig_group']]
      .sort_values('coef', ascending=False).to_string(index=False))